In [1]:
# %% Cell 1: Imports and Setup
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import os
import json
from datetime import datetime

# Seed everything for reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)


In [2]:

# %% Cell 2: Data Loading and Preprocessing (Run Once)
def load_and_preprocess_data(file_path, target_pollutant):
    """
    Load dataset and prepare for time series forecasting
    Returns scaled data with timestep structuring
    """
    # Load data
    df = pd.read_csv(file_path, parse_dates=['Timestamp'], index_col='Timestamp')
    
    # Select target pollutant and relevant features
    pollutants = [
        'PM2.5 (µg/m³)', 'PM10 (µg/m³)', 'NO (µg/m³)', 
        'NO2 (µg/m³)', 'SO2 (µg/m³)', 'CO (mg/m³)', 
        'Ozone (µg/m³)'
    ]
    df = df[pollutants].resample('15T').mean().ffill()
    
    # Create sliding window dataset
    def create_dataset(data, n_steps=1):
        X, y = [], []
        for i in range(len(data)-n_steps):
            X.append(data[i:(i+n_steps), :])
            y.append(data[i + n_steps, pollutants.index(target_pollutant)])
        return np.array(X), np.array(y)
    
    # Scale data
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(df)
    
    # Create time steps
    n_steps = 4  # 1 hour window (4*15min)
    X, y = create_dataset(scaled_data, n_steps)
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, shuffle=False, random_state=SEED
    )
    
    return (X_train, y_train), (X_test, y_test), scaler, df.shape[1]


In [3]:

# %% Cell 3: Model Architectures
def build_hybrid_model(input_shape, mlp_layers, lstm_layers, lstm_units, bidirectional=False):
    """
    Build hybrid MLP-LSTM model with configurable architecture
    """
    model = tf.keras.Sequential()
    
    # MLP Branch
    model.add(tf.keras.layers.Flatten(input_shape=input_shape))
    for units in mlp_layers:
        model.add(tf.keras.layers.Dense(units, activation='relu'))
        model.add(tf.keras.layers.Dropout(0.2))
    
    # Reshape for LSTM
    model.add(tf.keras.layers.Reshape((1, -1)))
    
    # LSTM Branch
    for i in range(lstm_layers):
        return_sequences = i < (lstm_layers - 1)
        if bidirectional:
            model.add(tf.keras.layers.Bidirectional(
                tf.keras.layers.LSTM(lstm_units, return_sequences=return_sequences)
            ))
        else:
            model.add(tf.keras.layers.LSTM(lstm_units, return_sequences=return_sequences))
        model.add(tf.keras.layers.Dropout(0.2))
    
    # Final Output
    model.add(tf.keras.layers.Dense(1))
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    return model


In [4]:

# %% Cell 4: Hyperparameter Configurations
POLLUTANTS = [
    'PM2.5 (µg/m³)', 'PM10 (µg/m³)', 'NO (µg/m³)',
    'NO2 (µg/m³)', 'SO2 (µg/m³)', 'CO (mg/m³)', 
    'Ozone (µg/m³)'
]

CONFIGURATIONS = [
    # MLP Layers, LSTM Layers, LSTM Units, Bidirectional
    ([64, 32], 1, 128, True),
    ([128], 2, 64, False),
    ([256, 128], 1, 256, True),
    ([64, 64], 2, 128, False),
    ([128, 64, 32], 1, 64, True),
]


In [5]:

# %% Cell 5: Training Loop with Checkpointing
def run_experiment(data_path, results_file='model_results.csv'):
    # Create results file if not exists
    if not os.path.exists(results_file):
        pd.DataFrame(columns=[
            'timestamp', 'pollutant', 'mlp_layers', 'lstm_layers',
            'lstm_units', 'bidirectional', 'mse', 'mae'
        ]).to_csv(results_file, index=False)
    
    for pollutant in POLLUTANTS:
        # Load data for current pollutant
        (X_train, y_train), (X_test, y_test), scaler, n_features = load_and_preprocess_data(
            data_path, pollutant
        )
        
        for config in CONFIGURATIONS:
            mlp_layers, lstm_layers, lstm_units, bidirectional = config
            
            # Skip already tested configurations
            existing = pd.read_csv(results_file)
            mask = (existing['pollutant'] == pollutant) & \
                   (existing['mlp_layers'].astype(str) == str(mlp_layers)) & \
                   (existing['lstm_layers'] == lstm_layers) & \
                   (existing['lstm_units'] == lstm_units) & \
                   (existing['bidirectional'] == bidirectional)
            if not mask.any():
                # Build and train model
                model = build_hybrid_model(
                    input_shape=(X_train.shape[1], X_train.shape[2]),
                    mlp_layers=mlp_layers,
                    lstm_layers=lstm_layers,
                    lstm_units=lstm_units,
                    bidirectional=bidirectional
                )
                
                early_stop = tf.keras.callbacks.EarlyStopping(
                    monitor='val_loss', patience=5, restore_best_weights=True
                )
                
                history = model.fit(
                    X_train, y_train,
                    validation_split=0.2,
                    epochs=100,
                    batch_size=32,
                    callbacks=[early_stop],
                    verbose=0
                )
                
                # Evaluate
                mse, mae = model.evaluate(X_test, y_test, verbose=0)
                
                # Save results
                new_row = pd.DataFrame([{
                    'timestamp': datetime.now().isoformat(),
                    'pollutant': pollutant,
                    'mlp_layers': str(mlp_layers),
                    'lstm_layers': lstm_layers,
                    'lstm_units': lstm_units,
                    'bidirectional': bidirectional,
                    'mse': mse,
                    'mae': mae
                }])
                
                new_row.to_csv(results_file, mode='a', header=False, index=False)
                print(f"Saved results for {pollutant} - {config}")


In [6]:

# %% Cell 6: Run the Experiment (Execute This Cell)
DATA_PATH = "Non_Null_Datasets/preprocessed_Raw_data_15Min_2024_site_5024_Alipur_Delhi_DPCC_15Min.csv"
run_experiment(DATA_PATH)


C:\Users\DELL\AppData\Local\Temp\ipykernel_26284\2707660949.py:16: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df = df[pollutants].resample('15T').mean().ffill()
c:\Users\DELL\Documents\Amrita\4th year\AirQuality_FederatedLearning\venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Saved results for PM2.5 (µg/m³) - ([64, 32], 1, 128, True)


c:\Users\DELL\Documents\Amrita\4th year\AirQuality_FederatedLearning\venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Saved results for PM2.5 (µg/m³) - ([128], 2, 64, False)


c:\Users\DELL\Documents\Amrita\4th year\AirQuality_FederatedLearning\venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


...

Saved results for Ozone (µg/m³) - ([256, 128], 1, 256, True)


c:\Users\DELL\Documents\Amrita\4th year\AirQuality_FederatedLearning\venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Saved results for Ozone (µg/m³) - ([64, 64], 2, 128, False)


c:\Users\DELL\Documents\Amrita\4th year\AirQuality_FederatedLearning\venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Saved results for Ozone (µg/m³) - ([128, 64, 32], 1, 64, True)


In [7]:

# %% Cell 7: Generate Report (Run After Experiment)
def generate_report(results_file='model_results.csv', report_file='model_report.csv'):
    results = pd.read_csv(results_file)
    report = []
    
    for pollutant in POLLUTANTS:
        subset = results[results['pollutant'] == pollutant]
        best_model = subset.loc[subset['mse'].idxmin()]
        
        report.append({
            'Pollutant': pollutant,
            'Best_Configuration': f"MLP{best_model['mlp_layers']} + {'Bi' if best_model['bidirectional'] else ''}LSTM({best_model['lstm_layers']}x{best_model['lstm_units']})",
            'MSE': best_model['mse'],
            'MAE': best_model['mae'],
            'Training_Date': best_model['timestamp']
        })
    
    report_df = pd.DataFrame(report)
    report_df.to_csv(report_file, index=False)
    return report_df

# Display report
generate_report()

,Pollutant,Best_Configuration,MSE,MAE,Training_Date
0,PM2.5 (µg/m³),MLP[128] + LSTM(2x64),0.000454,0.011243,2025-02-06T15:20:17.477448
1,PM10 (µg/m³),MLP[128] + LSTM(2x64),0.000563,0.011233,2025-02-06T15:22:53.657585
2,NO (µg/m³),"MLP[256, 128] + BiLSTM(1x256)",0.000074,0.003616,2025-02-06T15:27:24.358690
3,NO2 (µg/m³),"MLP[128, 64, 32] + BiLSTM(1x64)",0.000253,0.010107,2025-02-06T15:31:20.647216
4,SO2 (µg/m³),MLP[128] + LSTM(2x64),0.000060,0.005580,2025-02-06T15:32:25.773451
5,CO (mg/m³),MLP[128] + LSTM(2x64),0.000219,0.010473,2025-02-06T15:36:34.038071
6,Ozone (µg/m³),MLP[128] + LSTM(2x64),0.000910,0.014499,2025-02-06T15:39:53.249990
